# Exploring Kratos Multiphysics meshes with pyramids-mesh

[**Kratos Multiphysics**](https://github.com/KratosMultiphysics/Kratos) is an open-source framework for building
multiphysics finite-element (FE) and finite-volume (FV) solvers — structural mechanics, CFD, convection–diffusion,
shallow water, contact, poromechanics and more all ship as applications on the same core. Each solver reads its
geometry from a `.mdpa` file (**M**odel **P**art **D**ata): a plain-text format with `Begin Nodes … End Nodes` and
`Begin Elements <ElementType> … End Elements` blocks, where the element name encodes the topology (`Element2D3N` =
a 2-D 3-node triangle, `Element2D4N` = a quad, `Element3D4N` = a tetrahedron, `SmallDisplacementElement3D8N` = a
hexahedron, and so on).

`pyramids-mesh` is a *file-format reader*. It parses the **surface / area element blocks** of an `.mdpa` file —
the same reader handles files from dozens of different Kratos physics applications — and hands back a
`pyramids.netcdf.ugrid.MeshDataset`, the identical object pyramids produces from a NetCDF UGRID file. From there
the whole pyramids GIS toolbox applies. `pyramids-mesh` itself adds only the driver; every mesh operation lives
upstream in `pyramids`.

## 1. Imports

`MeshDataset` is the object every driver returns. Importing from `pyramids` also puts its vendored GDAL/PROJ data
in place, so nothing here is order-sensitive in a notebook — just import the names you use. `add_tiles` adds a web
basemap *only when the mesh is georeferenced*; engineering meshes are not, so we call it behind an `if ds.epsg`
guard throughout.

In [ ]:
from pyramids.netcdf.ugrid import MeshDataset
import numpy as np
import pandas as pd
from cleopatra.basemap.tiles import add_tiles

## 2. Locate the example data

This notebook lives in `docs/examples/real-data/`, so the repo-root `examples/data/real-data/` folder is **three**
levels up. Paths are written relative to the notebook itself, and `read_file` expects a `pathlib.Path`, never
`str(...)`. We start with a small **shallow-water** test mesh: a parabola over a rectangular channel.

In [ ]:
from pathlib import Path
DATA = Path("../../../examples/data/real-data")
tri_file = DATA / "kratos" / "shallow_water_OK_2Dtri_Element2D3N_205n320e_parabola-rectangle-0.25.mdpa"
tri_file, tri_file.exists()

## 3. Read a 2-D triangular mesh

`MeshDataset.read_file(path, driver=...)` parses the `Begin Nodes` / `Begin Elements` blocks and returns a
`MeshDataset` — the same type pyramids produces from a NetCDF UGRID file. We name the `"kratos"` driver explicitly
and pass the **`Path`** object.

In [ ]:
ds = MeshDataset.read_file(tri_file, driver="kratos")
type(ds).__name__

Now that a mesh has been read, cleopatra has switched matplotlib to a headless backend. Call its chooser once so
every later figure embeds inline in the notebook (and falls back to `Agg` in a plain script or in CI).

In [ ]:
from cleopatra.config import Config
Config.set_matplotlib_backend()

## 4. Explore the dataset

A `MeshDataset` exposes summary properties at a glance: `n_node` / `n_face` (mesh size), `crs` / `epsg`
(georeferencing, if the format recorded any), `bounds` (the `(minx, miny, maxx, maxy)` extent) and
`data_variable_names` (attached field variables).

In [ ]:
pd.Series({
    "n_node": ds.n_node,
    "n_face": ds.n_face,
    "crs": ds.crs,
    "epsg": ds.epsg,
    "bounds": tuple(round(float(b), 3) for b in ds.bounds),
    "data_variable_names": ds.data_variable_names,
})

205 nodes and 320 cells — a small shallow-water grid. `crs` / `epsg` are `None`: a Kratos model part lives in
engineering (simulation) units, not a geographic CRS. The extent runs from `(0, 0)` to `(10, 1)` — a long, thin
`10 × 1` rectangular channel. `data_variable_names` is empty: a 2-D `.mdpa` file is the grid only, with no third
coordinate and no solver fields (those come from the solver's output, not the mesh).

## 5. Connectivity: what shape are the cells?

`ds.mesh.face_node_connectivity.data` is the `(n_face, k)` table of node indices per face, where `k` is the widest
polygon; narrower cells are padded with `-1` (fill). Counting the non-negative entries per row gives each face's
**degree** — its number of nodes.

In [ ]:
conn = ds.mesh.face_node_connectivity.data
degree = (conn >= 0).sum(axis=1)
vals, counts = np.unique(degree, return_counts=True)
pd.Series(dict(zip(vals.tolist(), counts.tolist())), name="n_faces").rename_axis("nodes_per_face")

Every one of the 320 cells has **3 nodes** — a pure **triangular** mesh (`Element2D3N`). That matters for the GIS
section below: some geometric constructions (circumcenters) are defined only for triangles.

## 6. Wireframe: `plot_outline()`

The simplest view needs no data — just the mesh edges. `plot_outline()` draws the wireframe through cleopatra and
embeds it inline. The returned `MeshGlyph` wraps the matplotlib figure, so we can hand its axis to `add_tiles`
*if* the mesh were georeferenced (it is not, so the guard skips it).

In [ ]:
_g = ds.plot_outline()
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

The long `10 × 1` channel appears as a band of triangles — the discretised domain the shallow-water solver runs
on. This single read-and-draw is the whole point of the driver: a solver grid rendered with the same `MeshDataset`
call as any other format.

## 7. A data map: synthesize and colour a per-cell field

To draw a *filled* map we need a value per cell. The mesh carries geometry only, so we compute one — each cell's
**area** — and attach it with `with_variable(..., location="face")`, which returns a **new** dataset (the original
`ds` is untouched). Then `plot` colours the mesh by it.

In [ ]:
d2 = ds.with_variable("cell_area", ds.mesh.face_areas, location="face")
_g = d2.plot("cell_area", cmap="viridis", colorbar=True, title="Per-cell area (shallow-water channel)")
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

The colour map shows how the mesh is graded — larger triangles where the solution is smooth, smaller ones where
the parabola bends most sharply. This is the generic pyramids data-map path: any per-cell field (water depth,
pressure, temperature) plots in exactly the same way once it is attached as a face variable.

With a field attached, `.summary()` reports per-variable statistics — count, min, max, mean and std — one row per
data variable. It confirms the spread of cell areas numerically.

In [ ]:
d2.summary()

## 8. GIS operations, for free

Because the mesh is now a first-class pyramids object, the pyramids GIS layer applies directly — `pyramids-mesh`
itself adds none of this. Each cell below runs one operation and interprets it.

### Faces as a GeoDataFrame

`to_geodataframe()` turns every face into a polygon row of a geopandas `GeoDataFrame` — the standard vector
structure for export to shapefile / GeoPackage or for spatial joins. One row per triangle. Because the mesh has
`crs=None`, the frame is unprojected (model units, not metres or degrees).

In [ ]:
gdf = ds.to_geodataframe()
pd.Series({"rows": len(gdf), "columns": list(gdf.columns), "geom_type": gdf.geom_type.iloc[0]})

### Bounding polygon

`bounding_polygon()` dissolves the cells into a single shapely polygon of the meshed footprint — the gateway to
clipping and spatial joins. For this channel it is the `10 × 1` rectangle.

In [ ]:
poly = ds.bounding_polygon()
pd.Series({"geom_type": poly.geom_type, "bounds": tuple(round(float(b), 3) for b in poly.bounds),
           "area": round(float(poly.area), 3)})

### Circumcenters — defined for triangles

`circumcenters()` returns one circumcentre per cell. It is defined **only for triangular meshes** — a triangle has
a unique circumscribed circle, a general quad does not. This mesh is all triangles, so the call succeeds: one
`(x, y)` per face. These points are what a finite-volume scheme uses to build orthogonal dual cells.

In [ ]:
cc = ds.circumcenters()
pd.Series({"shape": cc.shape, "first_point": tuple(np.round(cc[0], 4).tolist())})

### Connected components

`connected_components()` labels each face by its face-adjacent group. A single label across all faces means the
mesh is one topologically connected piece — as a single solver domain should be.

In [ ]:
comp = ds.connected_components()
pd.Series({"n_components": int(comp.max()) + 1, "labels_seen": np.unique(comp).tolist()})

## 9. A topology tour across physics applications

The same `MeshDataset.read_file(path, driver="kratos")` call reads `.mdpa` files from many different Kratos
applications, with different element shapes. Below we read four representatives — a 2-D triangle mesh
(conjugate-heat-transfer), a 2-D quad mesh (contact mechanics), a 3-D-surface tetrahedral mesh (remeshing) and a
3-D-surface hexahedral mesh (structural mechanics) — and tabulate their size and element width. The reader builds
a planar `Mesh2d` in every case; the 3-D files additionally carry the node `z` coordinate as a `node_z` data
variable.

In [ ]:
tour = {
    "2D triangle (heat)":  "conjugate_heat_transfer_COMMENT_2Dtri_Element2D3N_661n1ke_cylinder-cooling-re100-pr2-test-solid.mdpa",
    "2D quad (contact)":   "contact_structural_mechanics_COMMENT_2Dquad_UpdatedLagrangianElement2D4N_100n69e_press-fit-2d.mdpa",
    "3D tet (remeshing)":  "mmg_remeshing_examples_COMMENT_3Dtet_Element3D4N_85n249e_coarse-sphere.mdpa",
    "3D hex (structural)": "structural_mechanics_COMMENT_3Dhex_SmallDisplacementElement3D8N_2kn2ke_tensile-test-example-mesh-hexahedra.mdpa",
}
rows = []
for label, name in tour.items():
    d = MeshDataset.read_file(DATA / "kratos" / name, driver="kratos")
    rows.append({
        "mesh": label,
        "n_node": d.n_node,
        "n_face": d.n_face,
        "nodes_per_cell": d.mesh.face_node_connectivity.data.shape[1],
        "has node_z": "node_z" in d.data_variable_names,
    })
pd.DataFrame(rows).set_index("mesh")

Four different applications, one reader. The connectivity width tracks the element: 3 for triangles, 4 for quads
and tetrahedra, 8 for hexahedra. Only the 3-D meshes carry `node_z` — the 2-D ones are flat in the `x–y` plane, so
there is no third coordinate to record.

To *see* one of the 3-D meshes, we draw the tetrahedral sphere's surface as a wireframe — the same `plot_outline()`
used for the flat channel, now showing a curved 3-D-surface mesh projected to the `x–y` plane.

In [ ]:
sphere = MeshDataset.read_file(
    DATA / "kratos" / "mmg_remeshing_examples_COMMENT_3Dtet_Element3D4N_85n249e_coarse-sphere.mdpa",
    driver="kratos",
)
_g = sphere.plot_outline()
if sphere.epsg:
    add_tiles(_g.ax, crs=sphere.epsg)

The circular outline is the coarse sphere seen from above — its surface triangles projected onto the plane. The
reader turned a remeshing-application `.mdpa` into the same kind of object as the shallow-water channel.

## 10. The limit: files that are not surface meshes

Not every `.mdpa` describes a 2-D surface. Kratos also stores **1-D** structural members (trusses, beams), **0-D**
particle sets (DEM — discrete-element spheres) and model parts with **no mesh at all** (data-only). `pyramids-mesh`
reads surface / area meshes, so it **declines** these with an explicit `UnsupportedMeshFormatError` (a subclass of
pyramids' `FileFormatNotSupportedError`) rather than silently producing a broken mesh. We loop over three real
rejected files, catch the error, and tabulate what each reported.

In [ ]:
from pyramids.netcdf.ugrid._drivers import UnsupportedMeshFormatError

rejected = {
    "line-only (truss)":   "structural_mechanics_COMMENT_2Dline_TrussElement3D2N_2n1e_truss-snap-through-snapthrough.mdpa",
    "point (DEM particles)": "dem_NOCOUNT_3Dpoint_SphericParticle3D_6kn6ke_bucket-non-uniform-2.gid-2dem.mdpa",
    "empty (no mesh)":     "dem_NOMESH_2Dempty_ModelPartData_empty_swimming-dem-updated.gid-updateddem-fem-bounda.mdpa",
}
rows = []
for label, name in rejected.items():
    try:
        MeshDataset.read_file(DATA / "kratos" / name, driver="kratos")
        rows.append({"file": label, "raised": "(none - read!)"})
    except UnsupportedMeshFormatError as e:
        rows.append({"file": label, "raised": type(e).__name__})
pd.DataFrame(rows).set_index("file")

All three are refused with the same `UnsupportedMeshFormatError`. The rejection is driven by the *element block* in
the file — the one-node, two-node or absent elements — not by the filename. The messages are deliberately explicit;
here is what each one tells the user.

In [ ]:
import re

def reason(err):
    "Strip the repeated file path and any Error(...) wrapper from the driver message."
    tail = str(err).rsplit(".mdpa", 1)[-1].lstrip(": '\"")
    m = re.match(r"[A-Za-z]+Error\(['\"](.*)['\"]\)\s*$", tail)
    return (m.group(1) if m else tail).rstrip("'\")").strip()

for label, name in rejected.items():
    try:
        MeshDataset.read_file(DATA / "kratos" / name, driver="kratos")
    except UnsupportedMeshFormatError as e:
        print(f"{label}:\n  {reason(e)}\n")

Each message names the real cause:

- **Line-only** — the truss file's elements have two nodes, and a surface *face* needs at least three; a 1-D
  topology belongs in edges or a network, not in faces.
- **Point (DEM)** — the file declares only one-node elements (discrete-element particles / point loads); a point
  cloud is not a mesh.
- **Empty** — the model part declares no `Nodes` at all, so there is nothing to read.

Declining is the right behaviour: flattening a 1-D, 0-D or empty model part into a `Mesh2d` would fabricate
topology that is not in the file. Extracting a surface from, say, a volume or particle model is a mesh *operation*
that belongs upstream in `pyramids`, not in a format reader.

## Takeaway

- **Kratos `.mdpa` = native multiphysics model part**: text `Nodes` / `Elements` blocks whose element names encode
  the topology, read into a `MeshDataset` with `MeshDataset.read_file(path, driver="kratos")`.
- **One reader, many applications**: the same call read shallow-water, conjugate-heat-transfer, contact, remeshing
  and structural meshes — triangles, quads, tetrahedra and hexahedra alike. 3-D meshes carry the `node_z`
  coordinate as a data variable; 2-D ones are flat.
- **The full pyramids GIS toolbox applies for free**: per-cell data maps, `to_geodataframe`, `bounding_polygon`,
  `circumcenters` (triangles only) and `connected_components` all work on the solver grid — `pyramids-mesh`
  contributes only the driver.
- **No CRS**: engineering meshes carry simulation coordinates (`crs` / `epsg` are `None`), so reprojection and
  basemap tiles are skipped.
- **Non-surface model parts are declined by design**: line-only (truss), point (DEM particle) and empty files each
  raise an `UnsupportedMeshFormatError` naming the real cause, rather than producing a broken mesh.